# Debug notebook — model + reward functions sanity check
Load a sample, run the model, run the reward functions, see everything.

In [1]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_PATH  = "/home/rlvr_experiments/outputs/sft_qwen_1.5b" #"Qwen/Qwen2.5-1.5B-Instruct"   # base or SFT checkpoint
TASK_NAME   = "countdown"                   # any reasoning-gym task
TASK_PARAMS = {"min_numbers": 2,"max_numbers": 3}                            # e.g. {"min_numbers": 4}
SEED        = 42
SAMPLE_IDX  = 0                             # which sample to inspect

MAX_NEW_TOKENS = 512
TEMPERATURE    = 0.7

In [2]:
import sys
sys.path.insert(0, "..")

import torch
import reasoning_gym
from transformers import AutoModelForCausalLM, AutoTokenizer
from environments import load_environment

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

/root/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


In [3]:
# ── Build environment (gives us reward functions + dataset) ───────────────────
config = {
    "environment": {
        "name": "reasoning_gym",
        "task_name": TASK_NAME,
        "samples": 50,
        "task_params": TASK_PARAMS,
    },
    "seed": SEED,
}

env = load_environment(config)
dataset = env.get_dataset(config)
reward_fns = env.get_reward_functions()

print(f"\nDataset size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"Reward functions: {[fn.__name__ for fn in reward_fns]}")

Generating 50 samples for 'countdown'...
  Task params: {'min_numbers': 2, 'max_numbers': 3}
  Generated 50 samples
  Sample answer: 4*95

Dataset size: 50
Columns: ['prompt', 'ground_truth', 'task', 'metadata']
Reward functions: ['correctness_reward']


In [4]:
# ── Inspect a sample ──────────────────────────────────────────────────────────
sample = dataset[SAMPLE_IDX]
print("prompt (messages):")
for msg in sample['prompt']:
    print(f"  [{msg['role']}] {msg['content']}")
print(f"\nground_truth: {sample['ground_truth']}")
print(f"metadata:     {sample.get('metadata', {})}")

prompt (messages):
  [user] Using all the numbers 95, 4, create an expression that equals 380.
You can only use each number once.

Final answer format instructions:
1. Provide your solution as a arithmetic expression (no '=' sign).
2. Do not include the target number in the expression.
3. Use '*' for multiplication.
4. Use '/' for division.
5. Do not include any other text or formatting.


ground_truth: 4*95
metadata:     {'difficulty': {'numbers': [2, 3], 'target': [100, 999], 'value': [1, 100]}, 'expression': '4*95', 'numbers': [95, 4], 'source_dataset': 'countdown', 'source_index': 0, 'target': 380}


In [5]:
# ── Load model ────────────────────────────────────────────────────────────────
print(f"Loading: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()
print("done")

Loading: /home/rlvr_experiments/outputs/sft_qwen_1.5b


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:03<00:00, 105.29it/s, Materializing param=model.norm.weight]                              


done


In [6]:
# ── Run model ─────────────────────────────────────────────────────────────────
system_prompt =(
    "You are a helpful assistant. You must always think step-by-step. Put your thinking process inside <think> and </think> tags, and put your final answer inside <answer> and </answer> tags."
)

# build messages: add system prompt + the user turn from the dataset
messages = [{"role": "system", "content": system_prompt}] + sample['prompt']

prompt_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        do_sample=TEMPERATURE > 0,
        pad_token_id=tokenizer.pad_token_id,
    )

new_tokens = out_ids[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=False)

print("=" * 60)
print("RAW MODEL OUTPUT")
print("=" * 60)
print(response)

RAW MODEL OUTPUT
<think>
To get to 380 from 95 and 4, I need to multiply 95 by 4 first because 95 * 4 = 380.
</think>
<answer>
95 * 4
</answer><|im_end|>


## Reward function evaluation

In [7]:
# ── Run reward functions on the model response ────────────────────────────────
# Reward functions receive completions as list-of-message-dicts (TRL format)
completion_wrapped = [[{"role": "assistant", "content": response}]]
ground_truth_list  = [sample["ground_truth"]]
metadata_list      = [sample.get("metadata", {})]

print(f"Ground truth : {sample['ground_truth']}")
print(f"Response     :\n{response}\n")
print("=" * 60)

total = 0.0
for fn in reward_fns:
    import inspect
    sig = inspect.signature(fn)
    kwargs = {}
    if "ground_truth" in sig.parameters:
        kwargs["ground_truth"] = ground_truth_list
    if "metadata" in sig.parameters:
        kwargs["metadata"] = metadata_list

    scores = fn(completion_wrapped, **kwargs)
    score  = scores[0]
    total += score
    print(f"{fn.__name__:<30}  score = {score:.3f}")

print("=" * 60)
print(f"{'TOTAL':<30}  score = {total:.3f}")

Ground truth : 4*95
Response     :
<think>
To get to 380 from 95 and 4, I need to multiply 95 by 4 first because 95 * 4 = 380.
</think>
<answer>
95 * 4
</answer><|im_end|>

correctness_reward              score = 1.000
TOTAL                           score = 1.000


In [13]:
print(prompt_text)

A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: Using all the numbers 95, 4, create an expression that equals 380.
You can only use each number once.

Final answer format instructions:
1. Provide your solution as a arithmetic expression (no '=' sign).
2. Do not include the target number in the expression.
3. Use '*' for multiplication.
4. Use '/' for division.
5. Do not include any other text or formatting.

Assistant:

